# BMBT downstream eval: v1 vs BMBT, controlled LM comparison

The one unmeasured piece of BMBT's actual bet: does the grammar-first structure make a
language model better *per token*, not just tie on raw token count (fertility already ties,
see docs/known-issues.md).

Two small decoder-only Transformers, trained from scratch, IDENTICAL architecture and
hyperparameters, differing only in which tokenizer produced their input token stream.
Compared on held-out **bits-per-byte** (not raw perplexity - the two tokenizers have
different vocabularies, bits-per-byte is what makes the comparison fair).

Set `DRIVE_DIR` below to a folder in your own Drive before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/bmbt-downstream-eval'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
!git clone --depth 1 https://github.com/konkomaji/bornomala.git
%cd bornomala/bengali-tokenizer
!pip install -e . -q
!pip install datasets -q

## Step 1: assemble the real corpus (once, shared by both tokenizers)

Same literary-weighted source mix already used to train the tokenizers themselves
(`configs/bpe-64k.json`), at a smaller scale for a bounded Colab run. Held-out is real
Wikipedia articles after the tokenizers' own training range - genuinely disjoint, checked
against a real bug caught locally before this notebook existed (see docs/known-issues.md:
an early version sliced by line index instead of article index and badly overlapped train
and held-out; fixed and verified before this script was trusted).

In [ ]:
CORPUS_DIR = os.path.join(DRIVE_DIR, 'corpus')
if not os.path.exists(os.path.join(CORPUS_DIR, 'train.txt')):
    !python scripts/assemble_lm_corpus.py --out-dir "{CORPUS_DIR}" --train-lines 200000 --held-out-articles 3000
else:
    print('corpus already assembled in Drive, skipping (delete the folder to rebuild)')

## Step 2: tokenize with both tokenizers

The trained artifacts (`bn-bpe-64k`, `bmbt-64k`) are already in the repo you just cloned.

In [ ]:
TOKENS_V1 = os.path.join(DRIVE_DIR, 'tokens-v1')
TOKENS_BMBT = os.path.join(DRIVE_DIR, 'tokens-bmbt')

!python scripts/prepare_lm_tokens.py --corpus-dir "{CORPUS_DIR}" \
  --tokenizer artifacts/bn-bpe-64k --out-dir "{TOKENS_V1}"
!python scripts/prepare_lm_tokens.py --corpus-dir "{CORPUS_DIR}" \
  --tokenizer artifacts/bmbt-64k --bmbt --out-dir "{TOKENS_BMBT}"

## Step 3: train both LMs

IDENTICAL hyperparameters for both cells below - do not change one without the other, or the
comparison stops being controlled. Each resumes automatically if this cell is re-run after a
disconnect (same `--ckpt-dir`).

`--batch-size 16 --grad-accum-steps 4` gives the SAME effective batch size (64) as a plain
`--batch-size 64` would, but fits a T4: the 64k-vocab output head's logits tensor is
`batch_size * block_size * vocab_size` floats - at batch_size=64 that's ~4.2GB on its own
(`64*256*64000*4` bytes), enough by itself to OOM a T4 once the rest of the model's activations
and AdamW's optimizer state are also on the card (this is exactly what happened on the first
run - real OOM traceback, not a guess). Splitting the same effective batch into 4 smaller
micro-batches keeps every hyperparameter that matters for the comparison identical while
shrinking the one tensor that actually didn't fit.

In [ ]:
CKPT_V1 = os.path.join(DRIVE_DIR, 'ckpt-v1')
CKPT_BMBT = os.path.join(DRIVE_DIR, 'ckpt-bmbt')

COMMON_ARGS = (
    '--device cuda --block-size 256 --d-model 384 --nhead 6 --num-layers 6 --dim-ff 1536 '
    '--batch-size 16 --grad-accum-steps 4 --lr 3e-4 --max-steps 10000 '
    '--save-every 500 --eval-every 500 --log-every 50'
)

In [ ]:
!python scripts/train_lm.py --tokens-dir "{TOKENS_V1}" --ckpt-dir "{CKPT_V1}" {COMMON_ARGS}

In [ ]:
!python scripts/train_lm.py --tokens-dir "{TOKENS_BMBT}" --ckpt-dir "{CKPT_BMBT}" {COMMON_ARGS}

## Step 4: read the result

Both training cells above print `FINAL held-out bits-per-byte` at the end - that is the
number to compare. Lower is better (fewer bits needed to predict the held-out text). Report
both numbers honestly, whichever way they land - a tie here is as real a result as BMBT's
fertility tie was, not a failure to fix.